In [ ]:
#| default_exp features.fsc_binlevel

In [ ]:
#| export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()


In [ ]:
#| export
class FSCBinEvaluator(FeatureEvaluator):
    """Extracts genome-wide and per-chromosome fragment size configurations."""
    
    name = "FSC_binlevel"
    source_file = ".FSC.ontarget.parquet"
    tier = 1
    category = "fragmentation"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
            
            cols = set(df.columns)
            ratios = ["core_short_ratio", "mono_nucl_ratio", "long_ratio"]
            
            # Global median over all bins
            for r in ratios:
                if r in cols:
                    extracted[f"global_{r}_median"] = float(df[r].median())
                    
            if "normalized_depth" in cols:
                extracted["global_normalized_depth_median"] = float(df["normalized_depth"].median())

            # Per-chromosome aggregates
            if "chrom" in cols:
                chroms = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]
                for c in chroms:
                    sub = df[df["chrom"] == c]
                    if sub.empty:
                        continue
                    for r in ratios:
                        if r in cols:
                            extracted[f"{c}_{r}_median"] = float(sub[r].median())

            return extracted
        except Exception as e:
            log.warning("fsc_binlevel_extraction_failed", error=str(e))
            return {}


In [ ]:
#| test
evaluator = FSCBinEvaluator()
assert evaluator.name == "FSC_binlevel"
df_test = pd.DataFrame([{"chrom": "chr1", "core_short_ratio": 0.2}])
res = evaluator.extract(df_test)
assert "global_core_short_ratio_median" in res
assert "chr1_core_short_ratio_median" in res
